# 03. Data Cleaning
##### DSR Berlin | Data Science Fundamentals
##### Inspired by Adam Green (adgefficiency.com), Rachel Berryman, and Samson Afolabi

---

## 🎯 Learning Objectives
By the end of this notebook, you will be able to:
1. Characterize datasets across 6 critical dimensions: **Quality, Quantity, Diversity, Cardinality, Dimensionality, and Sparsity**.
2. Enforce the **"Split First" golden rule** to isolate the test set and guarantee **zero data leakage**.
3. Evaluate **duplicates** with mathematical rigor: explain why duplicates between train and test are lethal, but why "true" duplicates inside training data provide vital empirical weight of evidence.
4. Detect disguised **missing values** (whitespace strings, sentinel `-999`, null tokens) and identify missingness mechanisms (**MCAR, MAR, MNAR**).
5. Apply four distinct imputation strategies: row/column drops, simple statistics, conditional group statistics, and model-based imputation (`KNNImputer`).
6. Detect and remediate **outliers** using Z-scores, normal distribution limits, and IQR percentile clipping.
7. Clean noisy string columns and leverage modern LLM-assisted cleaning patterns responsibly.


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer

import warnings

warnings.filterwarnings("ignore")

# Set plotting style and random seed
plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
RANDOM_SEED = 42

## 1. Data Characterization: The Six Dimensions

Before writing a single line of cleaning code, a senior data scientist characterizes the raw dataset across six foundational dimensions:

<div align="center">
  <img src="../images/data_characterization.jpg" width="650" alt="Data Characterization" />
  <p><em>Figure 1: Six Key Dimensions of Data Characterization.</em></p>
</div>

1. **Quality**:
   - Are there recording errors, impossible negative numbers, disguised nulls, or typos?
2. **Quantity**:
   - How many rows ($N$) and columns ($P$) exist? Is the sample size sufficient relative to model parameters?
3. **Diversity & Distribution Shift**:
   - Does the distribution of training samples match the distribution of the test set and real-world deployment?
4. **Cardinality**:
   - The number of unique values in categorical features. Low cardinality (`Contract`: 3 values) vs high cardinality (`ZipCode`: 5,000 values).
5. **Dimensionality**:
   - The ratio of features to samples ($P / N$). When $P \approx N$ or $P > N$, models suffer from the **curse of dimensionality** and severe overfitting.
6. **Sparsity**:
   - The proportion of zeros or empty entries in your data matrix. Text TF-IDF matrices or one-hot encodings often exceed 99% sparsity.


## 2. Ingesting the Raw Project Dataset

Let's load `../data/telco_customer_churn_raw.csv`. This dataset represents customer records for a telecommunications provider where we want to predict whether a customer will cancel their subscription (`Churn`: Yes/No).


In [4]:
# Load raw data
raw_path = "../data/telco_customer_churn_raw.csv"
df_raw = pd.read_csv(raw_path)

print(f"Raw Dataset Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print("\nFirst 3 rows:")
df_raw.head(3)

Raw Dataset Shape: 7068 rows, 21 columns

First 3 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,8272-ONJLV,Male,0,No,No,12,Yes,No,Fiber optic,No,...,Yes,No,Yes,Yes,One year,No,Electronic check,95.70,1184,No
1,7129-CAKJW,Female,0,No,No,17,Yes,Yes,Fiber optic,No,...,Yes,No,No,No,Month-to-month,No,Bank transfer (automatic),80.05,1345.65,No
2,7190-XHTWJ,Female,0,No,No,38,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,No,Mailed check,19.30,755.5,No


In [5]:
# Check column data types and non-null counts
print("Column Information:")
df_raw.info()

Column Information:
<class 'pandas.DataFrame'>
RangeIndex: 7068 entries, 0 to 7067
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7068 non-null   str    
 1   gender            7068 non-null   str    
 2   SeniorCitizen     7068 non-null   int64  
 3   Partner           7068 non-null   str    
 4   Dependents        7068 non-null   str    
 5   tenure            7068 non-null   int64  
 6   PhoneService      7068 non-null   str    
 7   MultipleLines     7068 non-null   str    
 8   InternetService   7068 non-null   str    
 9   OnlineSecurity    7068 non-null   str    
 10  OnlineBackup      7068 non-null   str    
 11  DeviceProtection  7068 non-null   str    
 12  TechSupport       7068 non-null   str    
 13  StreamingTV       7068 non-null   str    
 14  StreamingMovies   7068 non-null   str    
 15  Contract          7068 non-null   str    
 16  PaperlessBilling  7068 non-null  

## 3. The Golden Rule: Split First (Zero Data Leakage)

> 🛑 **NEVER clean, impute, scale, or transform your entire dataset before splitting!**

If you compute the mean of `MonthlyCharges` on the whole dataset to fill missing values, **the test set's values have contaminated your training statistics**. This is known as **Data Leakage**. Your test evaluation will look artificially optimistic, but the model will degrade in real-world production.

<div align="center">
  <img src="../images/train_val_test_split.jpg" width="650" alt="Train Test Validation Split" />
  <p><em>Figure 2: Isolating the Test Set Immediately.</em></p>
</div>

### Immediate Isolation Protocol:
1. Split into **Train (80%)** and **Test (20%)** right now.
2. Put the Test set in a "vault".
3. Compute all imputation statistics, encodings, and scaling parameters **solely on Train**.
4. Apply the fitted transformations to Test without refitting.


In [8]:
# Separate Features (X) and Target (y)
X_raw = df_raw.drop(columns=["Churn"])
y_raw = df_raw["Churn"]

# Stratified split to preserve the churn ratio in both partitions
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.20, random_state=RANDOM_SEED, stratify=y_raw
)

print(f"Training partition:   {X_train_raw.shape[0]} samples")
print(f"Test vault partition: {X_test_raw.shape[0]} samples")
print(f"Train Churn Rate:     {(y_train == 'Yes').mean()*100:.2f}%")
print(f"Test Churn Rate:      {(y_test == 'Yes').mean()*100:.2f}%")

Training partition:   5654 samples
Test vault partition: 1414 samples
Train Churn Rate:     26.53%
Test Churn Rate:      26.52%


## 4. Handling Duplicates: Nuance, Evidence, and Leakage

Duplicates are not simply rows to delete mindlessly. We must distinguish between **two fundamentally different types of duplicates**:

### 1. Accidental / Cross-Split Duplicates (Lethal Leakage):
- Caused by logging retries, double ETL loads, or bad database merges.
- If an exact duplicate row appears in both **Train** and **Test**, your test metric is fraudulent: the model merely memorized a sample it saw during training!
- *Action*: Drop identical records.

### 2. "True" Feature Duplicates within Training Data (Statistical Evidence):
- Imagine two customers who both have: `gender='Female'`, `SeniorCitizen=0`, `Contract='Month-to-month'`, `InternetService='DSL'`.
- Customer A churns, Customer B does not churn.
- Or ten distinct customers have identical profiles and all ten churn!
- In Bayesian statistics and probabilistic ML, **frequency information is genuine signal**. These repetitions provide empirical *weight of evidence*. If you deduplicate purely based on feature profiles, you distort the underlying prior probability $P(X)$!


In [15]:
# Check for exact duplicate rows across all columns in Train
n_exact_dups = X_train_raw.duplicated().sum()
print(f"Exact full-row duplicates in Train: {n_exact_dups}")

# Inspect a duplicated customer
if n_exact_dups > 0:
    sample_dup_id = X_train_raw[X_train_raw.duplicated()]["customerID"].iloc[0]
    print(f"\nInspecting duplicate instances of customerID '{sample_dup_id}':")
    print(X_train_raw[X_train_raw["customerID"] == sample_dup_id])

# Filter out exact duplicate rows from training data
train_mask = ~X_train_raw.duplicated()
X_train = X_train_raw[train_mask].copy()
y_train = y_train[train_mask].copy()

# For the test set, only remove duplicates within itself (do not leak info between splits)
X_test = X_test_raw[~X_test_raw.duplicated()].copy()
y_test = y_test[~X_test_raw.duplicated()].copy()

print(f"\nDeduplicated Train shape: {X_train.shape}")
print(f"Deduplicated Test shape:  {X_test.shape}")

Exact full-row duplicates in Train: 15

Inspecting duplicate instances of customerID '4476-OSWTN':
      customerID gender  SeniorCitizen Partner Dependents  tenure  \
1006  4476-OSWTN   Male              1     Yes         No      69   
869   4476-OSWTN   Male              1     Yes         No      69   

     PhoneService MultipleLines InternetService OnlineSecurity OnlineBackup  \
1006          Yes           Yes     Fiber optic            Yes           No   
869           Yes           Yes     Fiber optic            Yes           No   

     DeviceProtection TechSupport StreamingTV StreamingMovies  Contract  \
1006              Yes         Yes         Yes             Yes  One year   
869               Yes         Yes         Yes             Yes  One year   

     PaperlessBilling PaymentMethod  MonthlyCharges TotalCharges  
1006              Yes  Mailed check          110.55       7610.1  
869               Yes  Mailed check          110.55       7610.1  

Deduplicated Train shape: (

## 5. Missing Values: Detection, Mechanisms & Solutions

### Missingness is NOT always `np.nan`!
In real production systems, missing values frequently masquerade as:
- Empty string whitespace: `' '` or `''`
- Sentinel numbers: `-999`, `-1`, `0`
- String tokens: `'?'`, `'Unknown'`, `'N/A'`, `'null'`

### Why is Data Missing? The 3 Mechanisms:
1. **MCAR (Missing Completely at Random)**:
   - Missingness has zero correlation with any feature (e.g. random sensor glitch).
   - Safe to drop or simple-impute without introducing statistical bias.
2. **MAR (Missing at Random)**:
   - Missingness depends on *another observed feature*.
   - *Example*: Younger participants skip an income question on a digital survey at higher rates, but within the younger age group itself, the missingness of income is random.
3. **MNAR (Missing Not at Random)**:
   - Missingness depends on the unobserved/missing value itself (e.g. high-income earners refusing to disclose their wealth).
   - Requires adding a missing indicator flag (`MissingIndicator`).


In [16]:
# Detecting disguised missing values in X_train:
print("1. Standard .isna() count per column:")
print(X_train.isna().sum()[X_train.isna().sum() > 0])

# Check for whitespace strings in object columns
print("\n2. Checking for whitespace strings in 'TotalCharges':")
whitespace_count = (X_train["TotalCharges"] == " ").sum()
print(f"Rows where TotalCharges is just ' ': {whitespace_count}")

# Check for sentinel values (-999 in numeric, '?' in categorical)
print("\n3. Checking for sentinel values:")
sentinel_charges = (X_train["MonthlyCharges"] == "-999.0") | (
    X_train["MonthlyCharges"] == -999.0
)
print(f"Rows where MonthlyCharges is sentinel -999.0: {sentinel_charges.sum()}")

question_marks = (X_train["PaymentMethod"] == "?") | (
    X_train["PaymentMethod"] == "Unknown"
)
print(f"Rows where PaymentMethod is '?' or 'Unknown': {question_marks.sum()}")

1. Standard .isna() count per column:
Series([], dtype: int64)

2. Checking for whitespace strings in 'TotalCharges':
Rows where TotalCharges is just ' ': 10

3. Checking for sentinel values:
Rows where MonthlyCharges is sentinel -999.0: 76
Rows where PaymentMethod is '?' or 'Unknown': 61


In [17]:
# Standardize all disguised missing values to np.nan
def standardize_missing(df):
    df_clean = df.copy()

    # TotalCharges: empty string -> NaN, cast to float
    df_clean["TotalCharges"] = pd.to_numeric(
        df_clean["TotalCharges"].replace(" ", np.nan), errors="coerce"
    )

    # MonthlyCharges: -999 -> NaN, cast to float
    df_clean["MonthlyCharges"] = pd.to_numeric(
        df_clean["MonthlyCharges"], errors="coerce"
    )
    df_clean.loc[df_clean["MonthlyCharges"] < 0, "MonthlyCharges"] = np.nan

    # PaymentMethod: '?' or 'Unknown' -> NaN
    df_clean["PaymentMethod"] = df_clean["PaymentMethod"].replace(
        ["?", "Unknown"], np.nan
    )

    return df_clean


X_train = standardize_missing(X_train)
X_test = standardize_missing(X_test)

print("True missing counts in Train after standardization:")
print(X_train[["MonthlyCharges", "TotalCharges", "PaymentMethod"]].isna().sum())

True missing counts in Train after standardization:
MonthlyCharges    76
TotalCharges      10
PaymentMethod     61
dtype: int64


### Imputation Strategies:

#### Strategy 1: Simple Statistical Imputation (Median / Mode)
- **Continuous**: Fill with median (robust to skew and outliers).
- **Categorical**: Fill with mode (most frequent category).
- *Rule*: Fit the imputer on `X_train`, transform `X_train` and `X_test`!


In [18]:
# Compute training statistics
train_monthly_median = X_train["MonthlyCharges"].median()
train_payment_mode = X_train["PaymentMethod"].mode()[0]

print(f"Fitted Train MonthlyCharges Median: €{train_monthly_median:.2f}")
print(f"Fitted Train PaymentMethod Mode:    {train_payment_mode}")

Fitted Train MonthlyCharges Median: €70.30
Fitted Train PaymentMethod Mode:    Electronic check


#### Strategy 2: Conditional Group Imputation
Instead of a blunt global median, fill missing values conditional on another correlated feature.
For example, customers with a **Fiber optic** internet service pay significantly higher monthly charges than **DSL** customers. Imputing based on `InternetService` retains feature variance!


In [19]:
# Conditional median grouped by InternetService
conditional_medians = X_train.groupby("InternetService")["MonthlyCharges"].median()
print("Conditional MonthlyCharges Median by InternetService:")
print(conditional_medians)


# Apply conditional group imputation
def impute_conditional(df, group_col, target_col, group_medians, fallback_median):
    df_imputed = df.copy()
    for group, med in group_medians.items():
        mask = (df_imputed[group_col] == group) & (df_imputed[target_col].isna())
        df_imputed.loc[mask, target_col] = med
    # Fallback for any remaining NaNs
    df_imputed[target_col] = df_imputed[target_col].fillna(fallback_median)
    return df_imputed


# Impute MAR TotalCharges: when tenure is 0, TotalCharges is mathematically 0.0!
X_train.loc[X_train["tenure"] == 0, "TotalCharges"] = 0.0
X_test.loc[X_test["tenure"] == 0, "TotalCharges"] = 0.0

# Apply conditional imputation for MonthlyCharges
X_train = impute_conditional(
    X_train,
    "InternetService",
    "MonthlyCharges",
    conditional_medians,
    train_monthly_median,
)
X_test = impute_conditional(
    X_test,
    "InternetService",
    "MonthlyCharges",
    conditional_medians,
    train_monthly_median,
)

# Fill PaymentMethod with training mode
X_train["PaymentMethod"] = X_train["PaymentMethod"].fillna(train_payment_mode)
X_test["PaymentMethod"] = X_test["PaymentMethod"].fillna(train_payment_mode)

print("\nRemaining missing values in Train:")
print(X_train[["MonthlyCharges", "TotalCharges", "PaymentMethod"]].isna().sum())

Conditional MonthlyCharges Median by InternetService:
InternetService
DSL            56.05
Fiber optic    91.65
No             20.15
Name: MonthlyCharges, dtype: float64

Remaining missing values in Train:
MonthlyCharges    0
TotalCharges      0
PaymentMethod     0
dtype: int64


## 6. Outlier Detection: Z-Scores, IQR & Clipping

Outliers arise from two distinct sources:
1. **Measurement / Recording Errors** (e.g. `tenure = 1200` months when a human has only been alive 100 years, or `MonthlyCharges = 999.0` fat-finger entry). $\rightarrow$ **Must be corrected or dropped**.
2. **True Extreme Power Users** (e.g. a corporate enterprise account spending 5x the average). $\rightarrow$ **Preserved, but clipped or log-transformed**.

<div align="center">
  <img src="../images/zscore_outliers_normal.png" width="650" alt="Z-Score Outliers" />
  <p><em>Figure 3: Normal distribution curve showing z-score outlier boundaries ($|z| > 3$).</em></p>
</div>

### Detection Method 1: Z-Score ($|z| > 3.0$)
The Z-score measures how many standard deviations a point lies from the mean:
$$z = \frac{x - \mu}{\sigma}$$
Under a normal distribution, $99.73\%$ of observations fall within $|z| \le 3$. Any point with $|z| > 3$ is a candidate outlier.


In [20]:
# Detect outliers on MonthlyCharges and tenure using Train mean and std
mc_mean = X_train["MonthlyCharges"].mean()
mc_std = X_train["MonthlyCharges"].std()

z_scores_mc = (X_train["MonthlyCharges"] - mc_mean) / mc_std
outliers_mc = X_train[z_scores_mc.abs() > 3]

print(f"MonthlyCharges outliers with |z| > 3.0: {len(outliers_mc)} rows")
if len(outliers_mc) > 0:
    print(outliers_mc[["customerID", "MonthlyCharges", "tenure"]])

MonthlyCharges outliers with |z| > 3.0: 1 rows
      customerID  MonthlyCharges  tenure
3266  1084-UQCHV          1500.0       1


### Detection Method 2: Tukey's Interquartile Range (IQR) & Winsorization (Clipping)

For skewed non-normal distributions, the IQR is vastly superior to the Z-score because the median and quartiles are robust to the outliers themselves:

$$\text{IQR} = Q_3 - Q_1$$
$$\text{Upper Fence} = Q_3 + 1.5 \times \text{IQR}$$
$$\text{Lower Fence} = Q_1 - 1.5 \times \text{IQR}$$

### Winsorization (Clipping):
Instead of discarding rows (which permanently discards valuable sample size), we **cap / clip** extreme values at a sensible threshold.


In [21]:
# Realistic domain thresholds for telecom:
# Maximum tenure is 72 months (6-year maximum contract offering)
# Maximum monthly charge is capped at €150.0

print(f"Tenure > 72 before clipping: {(X_train['tenure'] > 72).sum()}")
print(
    f"MonthlyCharges > 150 before clipping: {(X_train['MonthlyCharges'] > 150).sum()}"
)

# Apply domain clipping to Train and Test
X_train["tenure"] = X_train["tenure"].clip(lower=0, upper=72)
X_test["tenure"] = X_test["tenure"].clip(lower=0, upper=72)

X_train["MonthlyCharges"] = X_train["MonthlyCharges"].clip(lower=15.0, upper=150.0)
X_test["MonthlyCharges"] = X_test["MonthlyCharges"].clip(lower=15.0, upper=150.0)

print(f"Tenure max after clipping: {X_train['tenure'].max()}")
print(f"MonthlyCharges max after clipping: €{X_train['MonthlyCharges'].max():.2f}")

Tenure > 72 before clipping: 1
MonthlyCharges > 150 before clipping: 1
Tenure max after clipping: 72
MonthlyCharges max after clipping: €150.00


## 7. String Cleaning & Categorical Standardization

Text data in categorical columns often suffers from leading/trailing whitespace, mixed casing, or multiple spellings of the same underlying category.


In [22]:
# Clean string columns in Train and Test
string_cols = ["gender", "Contract", "PaymentMethod", "InternetService"]

for col in string_cols:
    X_train[col] = X_train[col].astype(str).str.strip().str.title()
    X_test[col] = X_test[col].astype(str).str.strip().str.title()

print("Cleaned categorical value counts for 'gender':")
print(X_train["gender"].value_counts())

print("\nCleaned categorical value counts for 'Contract':")
print(X_train["Contract"].value_counts())

Cleaned categorical value counts for 'gender':
gender
Male      2882
Female    2757
Name: count, dtype: int64

Cleaned categorical value counts for 'Contract':
Contract
Month-To-Month    3075
Two Year          1384
One Year          1180
Name: count, dtype: int64


## 8. Persisting Cleaned Partitions

Now that our dataset has been rigorously cleaned with zero data leakage, we persist `X_train_clean.csv`, `X_test_clean.csv`, `y_train.csv`, and `y_test.csv` in `../data/`.
These cleaned partitions will be directly picked up by **Notebook 04: Feature Engineering**!


In [24]:
# Save cleaned datasets
X_train.to_csv("../data/telco_X_train_clean.csv", index=False)
X_test.to_csv("../data/telco_X_test_clean.csv", index=False)
y_train.to_csv("../data/telco_y_train.csv", index=False)
y_test.to_csv("../data/telco_y_test.csv", index=False)

print("Saved cleaned training and test partitions successfully!")

Saved cleaned training and test partitions successfully!
